In [1]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [2]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.6 MB/s eta 0:00:00a 0:00:01


In [3]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np

2026-05-22 11:16:25.615373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779448585.834206      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779448585.902899      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779448586.430021      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779448586.430062      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779448586.430064      57 computation_placer.cc:177] computation placer alr

In [4]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [5]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [6]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [7]:
tf_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            key:
            training_df[
                key
            ].values

            for key in
            training_df.columns
        }
    )
)

BATCH_SIZE = 4096

train = (
    tf_dataset
    .take(
        int(
            0.8 *
            len(training_df)
        )
    )
    .batch(BATCH_SIZE)
)

test = (
    tf_dataset
    .skip(
        int(
            0.8 *
            len(training_df)
        )
    )
    .batch(BATCH_SIZE)
)

2026-05-22 11:16:55.099049: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [9]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [10]:
#candidate dataset
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            "article_id":
                training_df[
                    "article_id"
                ],

            "product_type_name":
                training_df[
                    "product_type_name"
                ],

            "season":
                training_df[
                    "season"
                ],

            "purchase_count":
                training_df[
                    "purchase_count"
                ]
                .astype(float)
        }
    )
    .batch(256)
)

In [11]:
class TwoTowerModel(
    tfrs.models.Model
):

    def __init__(self):

        super().__init__()

        self.query_model = (
            QueryTower()
        )

        self.candidate_model = (
            CandidateTower()
        )

        self.task = (
            tfrs.tasks.Retrieval(
                metrics=
                tfrs.metrics.FactorizedTopK(

                    candidates=
                    candidate_dataset.map(
                        self.candidate_model
                    )
                )
            )
        )

    def compute_loss(
        self,
        features,
        training=False
    ):

        query_embeddings = (
            self.query_model(
                features
            )
        )

        candidate_embeddings = (
            self.candidate_model(
                features
            )
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )

In [12]:
model = (
    TwoTowerModel()
)

model.compile(
    optimizer=
    tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

In [15]:
for batch in train.take(1):

    model.compute_loss(
        batch,
        training=False
    )

print(
    "Model built!"
)

Model built!


In [16]:
model.load_weights(
    "/kaggle/input/datasets/selinparmar/two-tower-weights-model/two_tower_weights.weights.h5"
)

print(
    "Final model loaded!"
)

Final model loaded!


In [17]:
evaluation = model.evaluate(
    test,
    return_dict=True
)

evaluation

49/49 [==============================] - 5185s 106s/step - factorized_top_k/top_1_categorical_accuracy: 1.5000e-05 - factorized_top_k/top_5_categorical_accuracy: 1.5000e-05 - factorized_top_k/top_10_categorical_accuracy: 2.0000e-05 - factorized_top_k/top_50_categorical_accuracy: 6.0000e-05 - factorized_top_k/top_100_categorical_accuracy: 1.1500e-04 - loss: 3722619.2550 - regularization_loss: 0.0000e+00 - total_loss: 3722619.2550


{'factorized_top_k/top_1_categorical_accuracy': 1.4999999621068127e-05,
 'factorized_top_k/top_5_categorical_accuracy': 1.4999999621068127e-05,
 'factorized_top_k/top_10_categorical_accuracy': 1.9999999494757503e-05,
 'factorized_top_k/top_50_categorical_accuracy': 5.999999848427251e-05,
 'factorized_top_k/top_100_categorical_accuracy': 0.00011500000255182385,
 'loss': 2894651.5,
 'regularization_loss': 0.0,
 'total_loss': 2894651.5}

In [19]:
print(
    "Recall@1:",
    evaluation[
        'factorized_top_k/top_1_categorical_accuracy'
    ]
)

print(
    "Recall@5:",
    evaluation[
        'factorized_top_k/top_5_categorical_accuracy'
    ]
)

print(
    "Recall@10:",
    evaluation[
        'factorized_top_k/top_10_categorical_accuracy'
    ]
)

print(
    "Recall@50:",
    evaluation[
        'factorized_top_k/top_50_categorical_accuracy'
    ]
)

print(
    "Recall@100:",
    evaluation[
        'factorized_top_k/top_100_categorical_accuracy'
    ]
)

Recall@1: 1.4999999621068127e-05
Recall@5: 1.4999999621068127e-05
Recall@10: 1.9999999494757503e-05
Recall@50: 5.999999848427251e-05
Recall@100: 0.00011500000255182385


In [20]:
evaluation_results = {

    "Recall@1":
        evaluation[
            'factorized_top_k/top_1_categorical_accuracy'
        ],

    "Recall@5":
        evaluation[
            'factorized_top_k/top_5_categorical_accuracy'
        ],

    "Recall@10":
        evaluation[
            'factorized_top_k/top_10_categorical_accuracy'
        ],

    "Recall@50":
        evaluation[
            'factorized_top_k/top_50_categorical_accuracy'
        ],

    "Recall@100":
        evaluation[
            'factorized_top_k/top_100_categorical_accuracy'
        ],

    "Loss":
        evaluation[
            'loss'
        ]
}

evaluation_results

{'Recall@1': 1.4999999621068127e-05,
 'Recall@5': 1.4999999621068127e-05,
 'Recall@10': 1.9999999494757503e-05,
 'Recall@50': 5.999999848427251e-05,
 'Recall@100': 0.00011500000255182385,
 'Loss': 2894651.5}